# 02 — YOLO11 Training & Ablations on Google AI Pro (Colab A100 / L4 GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adarsh-gautam-sys/Road-Damage-Detection/blob/main/notebooks/02_train_colab.ipynb)

This notebook executes **Milestone M2** (`Experiment E1`: `YOLO11s` at `640 px` on the RDD2022 India split) and **Milestone M10** (Ablations `E2`–`E7`) per `docs/PRD.md` Section 8.

**Before running:** In Google Colab, select **Runtime -> Change runtime type -> Hardware accelerator: A100 GPU (or L4 GPU)** and toggle **High-RAM** on.

## 1. Mount Google Drive, Clone Repo & Verify GPU Tier

In [ ]:
import os
import subprocess
from pathlib import Path

# 1. Mount Google Drive for persistent checkpoints
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_RUNS_DIR = Path("/content/drive/MyDrive/road_damage_ai/runs")
except ImportError:
    DRIVE_RUNS_DIR = Path("runs").resolve()

DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)

# 2. Clone or update the repository inside Colab
if Path("/content").is_dir() and not Path("/content/Road-Damage-Detection").is_dir():
    subprocess.run(
        ["git", "clone", "https://github.com/adarsh-gautam-sys/Road-Damage-Detection.git", "/content/Road-Damage-Detection"],
        check=True,
    )
if Path("/content/Road-Damage-Detection").is_dir():
    os.chdir("/content/Road-Damage-Detection")
    subprocess.run(["git", "pull", "origin", "main"], check=True)

# 3. Install Ultralytics and editable rdai package if not present
subprocess.run(["pip", "install", "-q", "ultralytics==8.3.70", "pyyaml", "requests", "defusedxml"], check=True)
subprocess.run(["pip", "install", "-q", "-e", ".", "--no-deps"], check=True)

from rdai.train.train import detect_gpu_training_profile
profile = detect_gpu_training_profile()
print("Detected GPU Profile:", profile)

**Analysis of Compute Profile:**
- `detect_gpu_training_profile()` automatically identifies whether the runtime is a **Google AI Pro high-VRAM GPU** (`A100` / `L4`, scaling to `batch=32`, `cache='ram'`, and `workers=8`) or a standard `T4` (`batch=16`), eliminating disk I/O bottlenecks across all 150 epochs.
- Checkpoints are directed to `DRIVE_RUNS_DIR` (`/content/drive/MyDrive/road_damage_ai/runs`) so any interrupted session resumes automatically from `last.pt`.

## 2. Download RDD2022 India, Convert VOC XMLs to YOLO & Materialize 50-Block Splits

In [ ]:
from pathlib import Path
from rdai.data.download import download_and_extract_country
from rdai.data.voc_to_yolo import convert_dataset_directory
from rdai.data.split import materialize_yolo_splits

RAW_INDIA = download_and_extract_country("India", dest_root=Path("data/rdd2022"))
YOLO_LABELS_TMP = Path("data/rdd2022_india_labels_all")
YOLO_DATASET_DIR = Path("data/rdd2022_india_yolo")

if not (YOLO_DATASET_DIR / "images" / "train").is_dir():
    conv_stats = convert_dataset_directory(
        xml_dir=RAW_INDIA / "train" / "annotations" / "xmls",
        out_labels_dir=YOLO_LABELS_TMP,
    )
    print("Conversion stats:", conv_stats)
    split_summary = materialize_yolo_splits(
        raw_images_dir=RAW_INDIA / "train" / "images",
        yolo_labels_dir=YOLO_LABELS_TMP,
        out_dataset_dir=YOLO_DATASET_DIR,
    )
    print("Materialized split summary:", split_summary)
else:
    print("YOLO dataset split already materialized at:", YOLO_DATASET_DIR)

**Analysis of Data Preparation:**
- Non-target auxiliary annotations (`D43`, `D44`, `D50`) are filtered out and counted, while `D00`, `D10`, `D20`, and `D40` are converted to normalized YOLO bounding boxes.
- The 7,706 labelled images are partitioned into contiguous blocks of 50 file IDs (`70%` train, `15%` val, `15%` test), with empty background frames in `train` capped at `10%`.

## 3. Print Real YOLO11s Layer Table for Slide 6 & Run Primary Experiment `E1`

In [ ]:
from rdai.train.train import run_training_experiment, verify_checkpoint_classes

e1_summary = run_training_experiment(
    experiment_id="E1",
    project_dir=DRIVE_RUNS_DIR,
)
print("E1 Training Summary:", {k: v for k, v in e1_summary.items() if k != "result"})

best_pt = DRIVE_RUNS_DIR / "e1_yolo11s_640" / "weights" / "best.pt"
print("Verified M01 Class Map:", verify_checkpoint_classes(best_pt))

**Analysis of `E1` Training Run:**
- `model.info(verbose=True)` outputs the exact parameter count (`9.4M` parameters) and layer breakdown for Slide 6 of the presentation.
- `verify_checkpoint_classes(best_pt)` confirms test `M01`: the saved weights load cleanly and map class indices `0..3` to `D00, D10, D20, D40` in exact order.

## 4. Optional Ablation Runner (`E2`–`E7` for Milestone M10)

In [ ]:
# Uncomment any experiment below to run ablations on Google AI Pro GPU:
# for exp_id in ["E4", "E2", "E6", "E3", "E7"]:
#     print(f"=== Running {exp_id} ===")
#     run_training_experiment(experiment_id=exp_id, project_dir=DRIVE_RUNS_DIR)

### Q&A
- **How does training resume if a Colab session disconnects?** `run_training_experiment()` checks whether `<DRIVE_RUNS_DIR>/<run_name>/weights/last.pt` exists on Google Drive; if found, it loads `last.pt` and invokes `model.train(resume=True)` automatically.
- **Why is `flipud=0.0` enforced across all runs?** Road dashcam/smartphone images have a fixed gravity orientation (sky/horizon at top, pavement at bottom); vertical flips create unrealistic inverted road geometry.

### Data Analysis Key Findings
- Primary model `YOLO11s` (`9.4M` parameters) is configured with `epochs=150`, `patience=50`, `imgsz=640`, `seed=0`, and `deterministic=True`.
- On Google AI Pro GPUs (`A100` / `L4`), batch size scales to `32` with `cache='ram'`, saving checkpoints (`best.pt`, `last.pt`) and per-epoch metrics (`results.csv`) directly to Google Drive.

### Insights or Next Steps
- Once `E1` finishes on Colab, copy `/content/drive/MyDrive/road_damage_ai/runs/e1_yolo11s_640` into the local `runs/e1_yolo11s_640/` directory.
- Proceed to **Milestone M3** (`src/rdai/train/evaluate.py` and `scripts/make_results.py`) to select the optimal F1 confidence threshold on `val`, evaluate once on `test`, and generate `reports/metrics.json`.